# **Выбор слоя и верность карты: LayerCAM, HiResCAM, Grad-CAM++**

Практика к модулю [«Локализация и CAM-семейство»](https://ai-interpretability.school).

Урок утверждает три вещи, и все три можно не принимать на веру, а проверить числом:

1. Grad-CAM плох на ранних слоях, потому что усреднение градиента по позициям гасит его;
2. у HiResCAM сумма карты равна логиту класса с точностью до смещения;
3. pointing game меряет совпадение с человеческим ожиданием, а не верность модели.

Этим и займемся. Считает на процессоре, меньше минуты.

In [ ]:
import io
import urllib.request

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import models, transforms
from torchvision.models import ResNet50_Weights

torch.manual_seed(0)
DATA = 'https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main'

model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2).eval()

tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


def fetch(path):
    return urllib.request.urlopen(f'{DATA}/' + path, timeout=30).read()


img = tf(Image.open(io.BytesIO(fetch('data/cat_and_dog.jpg'))).convert('RGB')).unsqueeze(0)
names = [s.strip() for s in fetch('data/imagenet_classes.txt').decode().split('\n')]

LAYERS = {'layer2': model.layer2, 'layer3': model.layer3, 'layer4': model.layer4}

logits = model(img)
predicted = int(logits.argmax())
print(f'спрогнозированный класс: {predicted} {names[predicted]}')

## 1. Инструменты

Три метода отличаются одной строкой каждый — и в этом вся суть урока. Grad-CAM усредняет
градиент по позициям и получает **один вес на канал**. LayerCAM не усредняет вовсе: вес есть
у **каждого элемента**. HiResCAM умножает градиент на активацию **поэлементно**.

Обратите внимание на `relu=False` у `hires_cam`: он понадобится в разделе 3, и не просто так.

In [ ]:
def capture(layer, x, cls):
    """Активации слоя и градиент логита класса по ним — один прямой и один обратный проход."""
    store = {}
    handle = layer.register_forward_hook(lambda m, i, o: store.__setitem__('a', o))
    out = model(x)
    handle.remove()
    A = store['a']
    A.retain_grad()
    model.zero_grad()
    out[0, cls].backward()
    return A.detach(), A.grad.detach(), out.detach()


def grad_cam(A, G):
    """Grad-CAM: один вес на канал, усредненный градиент по всем позициям."""
    alpha = G.mean(dim=(2, 3), keepdim=True)
    return F.relu((alpha * A).sum(1))[0]


def layer_cam(A, G):
    """LayerCAM: вес на каждый элемент карты, усреднения нет."""
    return F.relu((F.relu(G) * A).sum(1))[0]


def hires_cam(A, G, relu=True):
    """HiResCAM: поэлементное произведение градиента на активацию."""
    m = (G * A).sum(1)[0]
    return F.relu(m) if relu else m


def upscale(cam):
    return F.interpolate(cam[None, None], (224, 224), mode='bilinear',
                         align_corners=False)[0, 0]

## 2. Почему Grad-CAM плох на ранних слоях

Урок объясняет это гашением: внутри одного канала градиент в одних местах положителен,
в других отрицателен, и при усреднении они уничтожают друг друга.

Проверить это можно прямо. Для каждого канала посчитаем две величины: модуль усредненного
градиента и усредненный модуль градиента. Их отношение и есть мера гашения:

- **близко к нулю** — градиенты внутри канала гасятся почти полностью;
- **единица** — гашения нет вовсе, знак по всему каналу один.

In [ ]:
print(f'слой     размер    каналов        гашение       весов почти 0')
for name, layer in LAYERS.items():
    A, G, _ = capture(layer, img, predicted)
    averaged = G.mean(dim=(2, 3))[0].abs()      # модуль усредненного градиента по каналу
    magnitude = G.abs().mean(dim=(2, 3))[0]     # усредненный модуль градиента по каналу
    ratio = (averaged / (magnitude + 1e-12)).mean().item()
    dead = (averaged < 0.01 * averaged.max()).float().mean().item()
    print(f'{name:9}{str(tuple(A.shape[-2:])):10}{A.shape[1]:<10}{ratio:12.3f}{dead*100:19.1f}%')

**Смотрите на строку `layer4`.** Отношение равно **ровно единице**, и это не
случайность и не округление. После последнего сверточного слоя в ResNet стоят global average
pooling и один линейный слой, поэтому

$$y^c = \sum_k w^c_k \frac{1}{Z}\sum_{ij} A^k_{ij} + b
\qquad\Longrightarrow\qquad
\frac{\partial y^c}{\partial A^k_{ij}} = \frac{w^c_k}{Z}$$

Градиент **не зависит от позиции** — он один и тот же во всей карте канала. Усреднять нечего,
гасить нечему. Вот почему «последний сверточный слой» для Grad-CAM не соглашение, а
единственный слой, где его усреднение ничего не теряет.

На `layer2` и `layer3` такого равенства нет, отношение падает — и вместе с ним теряется
пространственная информация. Ровно от этого и лечит LayerCAM.

**Задание 1.** Постройте карты обоих методов на всех трех слоях и посмотрите на них глазами.
Ячейка ниже это делает. Вопрос к вам: на каком слое разница между Grad-CAM и LayerCAM видна
сильнее всего, и совпадает ли это с тем, что показало отношение гашения?

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(11, 7))
for col, (name, layer) in enumerate(LAYERS.items()):
    A, G, _ = capture(layer, img, predicted)
    for row, (title, cam) in enumerate((('Grad-CAM', grad_cam(A, G)),
                                        ('LayerCAM', layer_cam(A, G)))):
        axes[row, col].imshow(upscale(cam).numpy(), cmap='jet')
        axes[row, col].set_title(f'{title} · {name}')
        axes[row, col].axis('off')
plt.tight_layout()
plt.show()

## 3. Гарантия HiResCAM: проверяем равенство

Урок утверждает: если после выбранного слоя идут только global average pooling и один
линейный слой, то **сумма всех элементов карты HiResCAM в точности равна логиту класса**
с точностью до смещения. Не приближенно — точно.

Такое утверждение либо проверяется, либо не стоит бумаги. Проверим.

In [ ]:
A, G, out = capture(model.layer4, img, predicted)
bias = model.fc.bias[predicted].item()
logit = out[0, predicted].item()

for label, relu in ((f'без ReLU', False), (f'с ReLU', True)):
    total = hires_cam(A, G, relu=relu).sum().item()
    print(f'{label:12} {total:9.4f} + {bias:7.4f} = {total + bias:9.4f}   '
          f'логит {logit:.4f}   расхождение {abs(total + bias - logit):.2e}')

**Расхождение порядка $10^{-7}$ — это ноль в точности чисел с плавающей точкой.**
Равенство держится.

А теперь посмотрите на вторую строку. **С ReLU равенство ломается.** И это не мелочь: формула
метода в уроке записана с ReLU, а гарантия верна для карты **до** него. ReLU обрезает
отрицательные вклады — те самые, которые говорят «против класса», — и сумма перестает
сходиться с логитом.

Так что точная формулировка такая: HiResCAM **раскладывает** логит, если брать карту без
ReLU. ReLU нужен, чтобы карту было видно глазами, и он же портит то самое свойство, ради
которого метод и брали. Держите это в голове, когда пишете в отчете «сумма карты равна логиту».

Убедимся, что дело не в удачном классе.

In [ ]:
weak = int(logits[0].argsort()[-200])
A, G, out = capture(model.layer4, img, weak)
bias, logit = model.fc.bias[weak].item(), out[0, weak].item()

print(f'слабый класс: {weak} {names[weak]}')
for label, relu in ((f'без ReLU', False), (f'с ReLU', True)):
    total = hires_cam(A, G, relu=relu).sum().item()
    print(f'{label:12} расхождение {abs(total + bias - logit):.3e}')

**Задание 2.** У слабого класса расхождение с ReLU на порядок больше. Объясните,
почему: что происходит с отрицательными вкладами, когда модель в классе не уверена?

Проверьте свою догадку — посчитайте, какая доля элементов карты отрицательна у сильного
класса и у слабого.

In [ ]:
# Ваш код здесь

## 4. Pointing game и ее главный изъян

Метрика простая: берем точку максимума карты и смотрим, попала ли она в рамку объекта.
На снимке кошка и собака, а модель уверенно видит собаку — посмотрим, куда укажет карта.

In [ ]:
# Рамки размечены на глаз по картинке 224 на 224 — это и есть та «человеческая разметка», о которой говорит урок
BOXES = {'кошка': (10, 40, 110, 200), 'собака': (115, 30, 215, 205)}

A, G, _ = capture(model.layer4, img, predicted)
peak = int(upscale(grad_cam(A, G)).argmax())
py, px = divmod(peak, 224)
print(f'максимум карты в точке ({px}, {py})')
for label, (x0, y0, x1, y1) in BOXES.items():
    hit = x0 <= px <= x1 and y0 <= py <= y1
    print(f'  {label:8} ({x0}, {y0})–({x1}, {y1})  попал' if hit
          else f'  {label:8} ({x0}, {y0})–({x1}, {y1})  мимо')

Максимум попал в рамку собаки и не попал в рамку кошки. Для класса-собаки это
«верно», для класса-кошки — «ошибка».

**А теперь главное.** Представьте, что модель узнает собаку в том числе по дивану, на котором
та лежит, — и это честная закономерность ее обучающих данных. Карта укажет на диван,
pointing game зачтет промах, и мы объявим метод плохим. Но ошиблись не метод и не модель:
ошиблись мы, предположив, что правильное объяснение обязано указывать на объект.

Именно поэтому pointing game — быстрый фильтр, а не критерий. Она сравнивает карту с
**нашим ожиданием**, а insertion и deletion спрашивают у самой модели.

**Задание 3.** Постройте карту для класса кошки и посчитайте pointing game для обеих рамок.
Получится, что для кошки метрика зачтет попадание, а для собаки — промах. Один и тот же
метод, одна и та же модель, разные ответы — от того, какой класс мы спросили. Что это
говорит о применимости метрики к сравнению **методов** между собой?

In [ ]:
# Ваш код здесь

## 5. Grad-CAM++ рядом с Grad-CAM

Grad-CAM++ перевзвешивает позиции по кривизне: точка, где рост активации быстро поднимает
оценку класса, получает больший вес. Урок обещает, что от этого маленький объект перестает
тонуть в усреднении.

Посмотрим на числах: сравним, насколько сосредоточены обе карты.

In [ ]:
def grad_cam_pp(A, G):
    """Grad-CAM++: вес позиции через кривизну. Множитель exp(логита) сокращается и в формуле не нужен."""
    g2, g3 = G ** 2, G ** 3
    denom = 2 * g2 + A.sum(dim=(2, 3), keepdim=True) * g3
    alpha = torch.where(denom != 0, g2 / denom, torch.zeros_like(denom))
    weights = (alpha * F.relu(G)).sum(dim=(2, 3), keepdim=True)
    return F.relu((weights * A).sum(1))[0]


def concentration(cam):
    """Доля массы карты в 10 % ярчайших элементов: чем выше, тем более сосредоточена карта."""
    values = cam.flatten()
    k = max(1, values.numel() // 10)
    return (torch.topk(values, k).values.sum() / (values.sum() + 1e-9)).item()


A, G, out = capture(model.layer4, img, predicted)
for title, cam in (('Grad-CAM', grad_cam(A, G)), ('Grad-CAM++', grad_cam_pp(A, G))):
    peak = int(upscale(cam).argmax())
    py, px = divmod(peak, 224)
    print(f'{title:11} максимум ({px:3}, {py:3})   доля массы в 10 % ярчайших {concentration(cam):.3f}')

**Задание 4.** У Grad-CAM++ карта менее сосредоточена — масса размазана шире.
Это ожидаемо: метод как раз и вытаскивает слабые отклики, которые Grad-CAM гасит усреднением.

Вопрос на подумать: означает ли «менее сосредоточена» — «хуже»? Проверьте это не глазами:
посчитайте для обеих карт deletion AUC (код есть в тетради `HW8`) и сравните числа.
Если Grad-CAM++ размазаннее, но deletion у него ниже, — какой из двух методов вы понесете
заказчице?

In [ ]:
# Ваш код здесь

## Что унести из тетради

- **«Последний сверточный слой» для Grad-CAM — не соглашение.** Это единственный слой, где
  градиент не зависит от позиции, а значит усреднение ничего не теряет. На ранних слоях
  отношение гашения падает, и вместе с ним теряется пространственная информация.
- **Гарантия HiResCAM настоящая — и хрупкая.** Сумма карты равна логиту до седьмого знака,
  но только без ReLU. С ReLU равенство ломается, и тем сильнее, чем менее уверена модель.
- **Pointing game отвечает не на тот вопрос.** Она меряет совпадение карты с нашей разметкой.
  Один и тот же метод получает и «верно», и «ошибка» в зависимости от того, какой класс
  спросили.
- **Разница между методами измерима.** Ни один вывод в этой тетради не сделан разглядыванием
  картинки, и это единственный способ выбирать метод, которому можно доверять.